In [1]:
import os
import pandas as pd

In [2]:
# Data from TexasMesonet: https://www.texasmesonet.org/DataProducts/CustomDownloads?


def load_rainfall(folder):
    combined = []
    for file in os.listdir(folder):
        df = pd.read_csv(os.path.join(folder, file))
        combined.append(df)

    df = pd.concat(combined)

    # Parse timestamps as naive 
    df['DateTime'] = pd.to_datetime(df[' Date_Time (UTC)'], utc=True)
    # Assume timzeone is correct
    # Make index timezone-naive but in Texas local time
    df.index = df['DateTime'].dt.tz_localize(None)

    df.sort_index(inplace=True)

    # Resample per hour
    return pd.DataFrame(df['Precipitation 1hr (in)'].resample('h').sum())
rain = load_rainfall("Precipitation")

In [3]:
rain

,Precipitation 1hr (in)
DateTime,
2020-01-01 20:00:00,0.001
2020-01-01 21:00:00,0.000
2020-01-01 22:00:00,0.000
2020-01-01 23:00:00,0.001
2020-01-02 00:00:00,0.110
...,...
2024-12-31 08:00:00,0.000
2024-12-31 09:00:00,0.000
2024-12-31 10:00:00,0.000


In [4]:
rain['Precipitation 1hr (in)'] = rain['Precipitation 1hr (in)'].fillna(0)


In [5]:
rain.to_csv("Precipitation_data.csv", index=True, index_label="DateTime")